In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from scipy import sparse


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("indic_language_dataset.csv")

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

df = df[
    [TEXT_COLUMN, LABEL_COLUMN]
].dropna()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str)


# ============================================================
# 2. TRAIN TEST SPLIT
# ============================================================

X_text = df[TEXT_COLUMN]
y = df[LABEL_COLUMN]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 3. COUNT VECTORIZER
# ============================================================

vectorizer = CountVectorizer()

X_train_count = vectorizer.fit_transform(
    X_train_text
)

X_test_count = vectorizer.transform(
    X_test_text
)

print("Training matrix:", X_train_count.shape)
print("Testing matrix :", X_test_count.shape)

print(
    "Non-zero training values:",
    X_train_count.nnz
)


# ============================================================
# 4. TF UNNORMALIZED
#
# TF_Un = -log(1 + nt) + 1
# ============================================================

def tf_unormalized(X):

    X = X.copy().astype(np.float64)

    X.data = -np.log(
        1 + X.data
    ) + 1

    return X


# ============================================================
# 5. TF NORMALIZATION 1
#
# TF_Nt1 = -log(1 + nt / N_d) + 1
#
# N_d = total number of words in sentence
# ============================================================

def tf_normalized_1(X):

    X = X.copy().astype(np.float64)

    total_words = np.asarray(
        X.sum(axis=1)
    ).ravel()

    total_words[
        total_words == 0
    ] = 1

    rows = np.repeat(
        np.arange(X.shape[0]),
        np.diff(X.indptr)
    )

    X.data = (
        -np.log(
            1 +
            X.data /
            total_words[rows]
        )
        + 1
    )

    return X


# ============================================================
# 6. TF NORMALIZATION 2
#
# TF_Nt2 = -log(1 + nt / max(n)) + 1
#
# max(n) = maximum word frequency in sentence
# ============================================================

def tf_normalized_2(X):

    X = X.copy().astype(np.float64)

    max_frequency = np.zeros(
        X.shape[0]
    )

    for i in range(X.shape[0]):

        start = X.indptr[i]
        end = X.indptr[i + 1]

        if start < end:
            max_frequency[i] = np.max(
                X.data[start:end]
            )

    max_frequency[
        max_frequency == 0
    ] = 1

    rows = np.repeat(
        np.arange(X.shape[0]),
        np.diff(X.indptr)
    )

    X.data = (
        -np.log(
            1 +
            X.data /
            max_frequency[rows]
        )
        + 1
    )

    return X


# ============================================================
# 7. DOCUMENT FREQUENCY
#
# nt = number of sentences containing the word
# ============================================================

nt = np.asarray(
    (X_train_count > 0).sum(axis=0)
).ravel()

N = X_train_count.shape[0]

print("Number of documents:", N)
print("Vocabulary size:", len(nt))


# ============================================================
# 8. IDF UNNORMALIZED
#
# IDF_Un = -log(N / nt + 1)
# ============================================================

idf_un = -np.log(
    (N / nt) + 1
)


# ============================================================
# 9. IDF NORMALIZED
#
# IDF_Nt1 =
# -log(max(n) / nt + 1) + 1
# ============================================================

max_n = np.max(nt)

idf_nt1 = -np.log(
    (max_n / nt) + 1
) + 1


# ============================================================
# 10. CALCULATE TF
# ============================================================

print("\nCalculating TF...")

tf_un_train = tf_unormalized(
    X_train_count
)

tf_nt1_train = tf_normalized_1(
    X_train_count
)

tf_nt2_train = tf_normalized_2(
    X_train_count
)


tf_un_test = tf_unormalized(
    X_test_count
)

tf_nt1_test = tf_normalized_1(
    X_test_count
)

tf_nt2_test = tf_normalized_2(
    X_test_count
)


print("TF calculation completed.")


# ============================================================
# 11. LOGISTIC REGRESSION FUNCTION
# ============================================================

def evaluate_combination(
    name,
    tf_train,
    tf_test,
    idf
):

    print(
        "\nRunning:",
        name
    )

    X_train = tf_train.multiply(
        idf
    )

    X_test = tf_test.multiply(
        idf
    )

    model = LogisticRegression(
        max_iter=2000,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print(
        "Accuracy :",
        round(accuracy, 4)
    )

    print(
        "Precision:",
        round(precision, 4)
    )

    print(
        "Recall   :",
        round(recall, 4)
    )

    print(
        "F1 Score :",
        round(f1, 4)
    )

    return {
        "Combination": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }


# ============================================================
# 12. RUN ALL SIX COMBINATIONS
# ============================================================

results = []


# ------------------------------------------------------------
# 1. Un-Un
# ------------------------------------------------------------

results.append(
    evaluate_combination(
        "Un-Un",
        tf_un_train,
        tf_un_test,
        idf_un
    )
)


# ------------------------------------------------------------
# 2. Nt1-Un
# ------------------------------------------------------------

results.append(
    evaluate_combination(
        "Nt1-Un",
        tf_nt1_train,
        tf_nt1_test,
        idf_un
    )
)


# ------------------------------------------------------------
# 3. Nt2-Un
# ------------------------------------------------------------

results.append(
    evaluate_combination(
        "Nt2-Un",
        tf_nt2_train,
        tf_nt2_test,
        idf_un
    )
)


# ------------------------------------------------------------
# 4. Un-Nt1
# ------------------------------------------------------------

results.append(
    evaluate_combination(
        "Un-Nt1",
        tf_un_train,
        tf_un_test,
        idf_nt1
    )
)


# ------------------------------------------------------------
# 5. Nt1-Nt1
# ------------------------------------------------------------

results.append(
    evaluate_combination(
        "Nt1-Nt1",
        tf_nt1_train,
        tf_nt1_test,
        idf_nt1
    )
)


# ------------------------------------------------------------
# 6. Nt2-Nt1
# ------------------------------------------------------------

results.append(
    evaluate_combination(
        "Nt2-Nt1",
        tf_nt2_train,
        tf_nt2_test,
        idf_nt1
    )
)


# ============================================================
# 13. FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 14. BEST COMBINATION
# ============================================================

best = results_df.loc[
    results_df["Accuracy"].idxmax()
]

print("\nBest combination:")
print(
    best["Combination"]
)

print(
    "Best accuracy:",
    round(
        best["Accuracy"],
        4
    )
)


# ============================================================
# 15. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "tf_idf_results.csv",
    index=False
)

print(
    "\nResults saved to tf_idf_results.csv"
)

Training matrix: (90576, 104320)
Testing matrix : (22644, 104320)
Non-zero training values: 960139
Number of documents: 90576
Vocabulary size: 104320

Calculating TF...
TF calculation completed.

Running: Un-Un
Accuracy : 0.8688
Precision: 0.8853
Recall   : 0.8688
F1 Score : 0.8746

Running: Nt1-Un
Accuracy : 0.869
Precision: 0.8869
Recall   : 0.869
F1 Score : 0.8752

Running: Nt2-Un
Accuracy : 0.8787
Precision: 0.8915
Recall   : 0.8787
F1 Score : 0.8832

Running: Un-Nt1
Accuracy : 0.8679
Precision: 0.8897
Recall   : 0.8679
F1 Score : 0.8754

Running: Nt1-Nt1
Accuracy : 0.8623
Precision: 0.8907
Recall   : 0.8623
F1 Score : 0.8721

Running: Nt2-Nt1
Accuracy : 0.8746
Precision: 0.8943
Recall   : 0.8746
F1 Score : 0.8814


FINAL RESULTS
Combination  Accuracy  Precision   Recall  F1 Score
      Un-Un  0.868751   0.885293 0.868751  0.874644
     Nt1-Un  0.869016   0.886923 0.869016  0.875235
     Nt2-Un  0.878688   0.891499 0.878688  0.883176
     Un-Nt1  0.867868   0.889651 0.867868  0.875

total sample: 113220,2
<p>vocab size= 104320
<p>Training matrix: (90576, 104320)
Testing matrix : (22644, 104320)

In [ ]:
X_train_count

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 960139 stored elements and shape (90576, 104320)>